In [ ]:
!pip install pgembed sqlalchemy psycopg2-binary sqlalchemy_utils

In [2]:
import pgembed
srv = pgembed.get_server('./mypgdata')

In [3]:
print(srv.psql('SELECT 1+1 as res;'))

 res 
-----
   2
(1 row)




In [4]:
from sqlalchemy_utils import create_database, database_exists
from sqlalchemy import create_engine
import sqlalchemy as sql

In [5]:
dburi = srv.get_uri(database='mydb')
display(dburi)
if not database_exists(dburi):
    create_database(dburi)
engine = create_engine(dburi)

'postgresql://postgres:@/mydb?host=/Users/orm/repos/pgembed/mypgdata'

In [6]:
table_name = 'mytable'
with engine.connect() as conn:
    conn.execute(sql.text(f"create table {table_name} (id int);"))
    conn.execute(sql.text(f"insert into {table_name} values (1);"))
    cur = conn.execute(sql.text(f"select * from {table_name};"))
    result = cur.fetchone()

In [7]:
result

(1,)

## BM25 full-text search with `pgembed_stannum`

The bundled [stannum](https://github.com/wuxianliang/stannum) extension adds BM25
full-text search (with a jieba tokenizer for Chinese) over plain `text` columns.
`pgembed_stannum.StannumIndex` manages one stannum index and searches it through
psycopg2 with bound parameters, so arbitrary agent queries are safe.

In [ ]:
from pgembed_stannum import StannumIndex, StannumRetriever

srv.psql('CREATE EXTENSION IF NOT EXISTS stannum')
srv.psql('CREATE EXTENSION IF NOT EXISTS vector')
srv.psql('''
CREATE TABLE IF NOT EXISTS articles (id int PRIMARY KEY, body text, embedding vector(3));
DELETE FROM articles;
INSERT INTO articles VALUES
 (1, 'PostgreSQL 数据库 内核优化', '[0.1, 0.0, 0.0]'),
 (2, 'The database query planner', '[0.8, 0.1, 0.0]'),
 (3, 'Storage engine internals', '[0.12, 0.01, 0.0]');
''')

index = StannumIndex(srv, 'articles', 'body')  # jieba tokenizer by default
index.create()
index.index_name

In [ ]:
hits = index.search('数据库', limit=3)
hits

In [ ]:
print('count:', index.search_count('database'))
print('analysis:', index.analysis())
print('health:', index.check_health())

# Plain-text mode: tokenizes the query with the index tokenizer and drops the
# built-in 'auto' stop words (的 is one), so the query reduces to 数据库.
print('stop words dropped:', index.search('的 数据库', drop_stop_words=True))

### Hybrid search (BM25 + vector, fused with reciprocal rank fusion)

`hybrid_search` fuses stannum BM25 ranking with pgvector cosine distance in a
single SQL statement. On small tables the planner may sequential-scan; at or
above `seqscan_row_threshold` rows it refuses to run without an applicable
index on the vector column.

In [ ]:
fused = index.hybrid_search(
    'database',
    vector_column='embedding',
    query_vector=[0.1, 0.0, 0.0],
    limit=3,
)
fused

### Retriever adapters (optional extras)

`StannumRetriever.for_langchain` / `for_llama_index` wrap the index in the
framework's retriever interface (`Document` / `NodeWithScore`). They need the
`pgembed-stannum[langchain]` or `pgembed-stannum[llama-index]` extra.

In [ ]:
try:
    retriever = StannumRetriever.for_langchain(index, k=3)
    retriever.invoke('database')
except ImportError as exc:
    print(exc)  # pip install 'pgembed-stannum[langchain]'